# Product Evidence Platform — Belief-Driven Identification and Mandatory Product URL

The workflow first internalizes `MAIN_TEXT + COUNTRY_CODE` without web evidence, constructs competing product hypotheses, and then searches through the immutable market route:

```text
requested retailer, when provided
→ alternative retailer within the requested country
→ global fallback
```

Every `COMPLETED` or `REVIEW_REQUIRED` run must deliver a real direct product URL. The final URL is browser-openable, product-detail-like, information-rich, and suitable for team eyeballing. If no safe direct product URL is found, the run terminates with `MANDATORY_PRODUCT_URL_NOT_FOUND` rather than fabricating a URL.


In [ ]:
from __future__ import annotations
import ast, importlib, json, subprocess, sys
from pathlib import Path
from pprint import pprint


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'docker-compose.yml').is_file() and (candidate / 'src' / 'product_evidence_harness').is_dir():
            return candidate
    raise RuntimeError('Could not locate the web_search_tool repository root')


PROJECT_ROOT = find_project_root()
SRC_PATH = (PROJECT_ROOT / 'src').resolve()
LOCAL_PACKAGE = (SRC_PATH / 'product_evidence_harness').resolve()
REQUIRED_MODULE = LOCAL_PACKAGE / 'notebook_runtime.py'
if not REQUIRED_MODULE.is_file():
    raise FileNotFoundError(f'Missing {REQUIRED_MODULE}. Pull the current repository branch and restart the kernel.')

for required_path in (str(PROJECT_ROOT), str(SRC_PATH)):
    sys.path[:] = [item for item in sys.path if str(Path(item or '.').resolve()) != required_path]
    sys.path.insert(0, required_path)

for module_name in tuple(sys.modules):
    if module_name == 'product_evidence_harness' or module_name.startswith('product_evidence_harness.') or module_name == 'src.product_evidence_harness' or module_name.startswith('src.product_evidence_harness.'):
        del sys.modules[module_name]
importlib.invalidate_caches()

PACKAGE_IMPORTS = {
    'pandas': 'pandas>=2.2,<3',
    'matplotlib': 'matplotlib>=3.8,<4',
    'seaborn': 'seaborn>=0.13.2,<1',
    'rich': 'rich>=13.7,<15',
    'openpyxl': 'openpyxl>=3.1,<4',
}
missing_specs = []
for module_name, package_spec in PACKAGE_IMPORTS.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        missing_specs.append(package_spec)
if missing_specs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_specs])

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display
from rich.console import Console
from rich.panel import Panel

import product_evidence_harness
loaded_package = Path(product_evidence_harness.__file__).resolve()
if LOCAL_PACKAGE not in loaded_package.parents:
    raise RuntimeError(f'Wrong package loaded. Expected {LOCAL_PACKAGE}; loaded {loaded_package}')

from product_evidence_harness.notebook_runtime import AGENT_URL, HEARTBEAT_SECONDS, DEFAULT_FEATURE_SET, check_health, run_product, host_artifact_dir
from product_evidence_harness.notebook_diagnostics import (
    build_single_product_diagnostics, display_compact, display_rich_summary,
    plot_all_diagnostics, plot_candidate_outcomes, plot_confidence_distribution,
    plot_confidence_vs_coverage, plot_domain_quality, plot_feature_heatmap,
    plot_funnel, plot_rejection_reasons, plot_stage_yield,
)
from product_evidence_harness.source_authority_notebook import apply_source_authority_notebook_patch
from product_evidence_harness.adaptive_notebook_diagnostics import (
    build_adaptive_search_diagnostics, display_adaptive_search_summary,
    export_adaptive_search_tables, plot_credit_progression,
    plot_engine_candidate_yield, plot_engine_credit_allocation,
)

apply_source_authority_notebook_patch()
sns.set_theme(context='notebook', style='whitegrid')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 180)
pd.set_option('display.width', 260)

health = check_health()
feature_sets = sorted(path.stem for path in (PROJECT_ROOT / 'inputs' / 'private').glob('*.json'))
if DEFAULT_FEATURE_SET not in feature_sets:
    raise RuntimeError('The committed inputs/private/toy_features.json is missing')
Console().print(Panel(
    f'Repository: {PROJECT_ROOT}\nPackage: {loaded_package}\nAgent: {AGENT_URL}\nDefault feature set: {DEFAULT_FEATURE_SET}\nAvailable feature sets: {", ".join(feature_sets)}',
    title='Notebook readiness',
))
pprint(health)


## 1. Run one product

`main_text` and `country_code` are mandatory. `retailer_name`, `ean`, and `language_code` are optional. Keep EAN/GTIN as text.

`RUN_SINGLE_PRODUCT` defaults to `False` to prevent accidental paid SerpAPI calls.


In [ ]:
FEATURE_SET = 'toy_features'
RUN_SINGLE_PRODUCT = False
product = {
    'row_id': 'ROW-001',
    'main_text': 'Replace with the vendor product main text',
    'country_code': 'CZ',
    'retailer_name': None,
    'ean': None,
    'language_code': None,
}

if RUN_SINGLE_PRODUCT:
    if product['main_text'].startswith('Replace with'):
        raise ValueError("Replace product['main_text'] before running")
    result = run_product(product, FEATURE_SET)
    search = result.get('search') or {}
    acceptance = result.get('primary_url_acceptance') or {}
    delivery = result.get('url_delivery') or {}
    if not result.get('primary_url') or not delivery.get('delivered'):
        raise RuntimeError(f'Mandatory URL contract violated: {json.dumps(result, indent=2, default=str)[:6000]}')
    pprint({
        'row_id': (result.get('product') or {}).get('row_id'),
        'job_status': result.get('job_status'),
        'primary_url': result.get('primary_url'),
        'url_delivery_status': delivery.get('status'),
        'strictly_verified': delivery.get('strictly_verified'),
        'strict_primary_accepted': acceptance.get('accepted'),
        'delivered': delivery.get('delivered'),
        'selection_scope': (result.get('product_match') or {}).get('selection_scope'),
        'engine_sequence': search.get('engine_sequence'),
        'target_source_tiers': search.get('target_source_tiers'),
        'market_decision_path': search.get('market_decision_path'),
        'serpapi_requests_used': search.get('serpapi_requests_used'),
        'search_stop_reason': search.get('stop_reason'),
        'product_identification': result.get('product_identification'),
    })
else:
    print('Ready. Replace the product input and set RUN_SINGLE_PRODUCT = True.')


## 2. Product understanding and belief state

This section shows the offline product interpretation, competing hypotheses, critical uncertainty, and evidence-driven probability updates. The Markdown artifacts are observable decision summaries, not hidden chain-of-thought.


In [ ]:
if 'result' not in globals():
    raise RuntimeError('Run the single-product cell first')
artifact_path = host_artifact_dir(PROJECT_ROOT, result)
if artifact_path is None or not artifact_path.is_dir():
    raise RuntimeError('Repository-local artifact directory was not found')
belief_path = artifact_path / 'product_belief.json'
if not belief_path.is_file():
    raise RuntimeError(f'Missing belief artifact: {belief_path}')
belief = json.loads(belief_path.read_text(encoding='utf-8'))
leading = belief.get('leading_hypothesis') or {}
metrics = belief.get('metrics') or {}
product_identification_df = pd.DataFrame([{'identified_product': leading.get('canonical_name'), 'resolution_status': belief.get('resolution_status'), 'leading_probability': leading.get('posterior_probability'), 'posterior_margin': metrics.get('posterior_margin'), 'search_readiness': metrics.get('search_readiness'), 'identity_completeness': metrics.get('identity_completeness'), 'ambiguity_entropy': metrics.get('ambiguity_entropy'), 'assumption_burden': metrics.get('assumption_burden'), 'interpretation_source': belief.get('interpretation_source'), 'evidence_items': len(belief.get('evidence_ledger') or [])}])
hypotheses_df = pd.DataFrame(belief.get('hypotheses') or [])
uncertainties_df = pd.DataFrame(belief.get('uncertainties') or [])
belief_updates_df = pd.DataFrame(belief.get('snapshots') or [])
evidence_ledger_df = pd.DataFrame(belief.get('evidence_ledger') or [])
display_compact(product_identification_df, title='Product identification summary', max_rows=5)
display_compact(hypotheses_df, title='Competing product hypotheses', max_rows=10)
display_compact(uncertainties_df, title='Decision-critical uncertainties', max_rows=20)
display_compact(belief_updates_df, title='Belief updates after evidence', max_rows=30)
display_compact(evidence_ledger_df, title='Atomic evidence ledger', max_rows=100)
for artifact_name in ('product_understanding.md', 'market_decision_path.md', 'belief_updates.md'):
    artifact_file = artifact_path / artifact_name
    if artifact_file.is_file():
        display(Markdown(artifact_file.read_text(encoding='utf-8')))


## 3. Complete diagnostic model


In [ ]:
diagnostics = build_single_product_diagnostics(result, artifact_dir=artifact_path)
adaptive_diagnostics = build_adaptive_search_diagnostics(result)
overview_df = diagnostics.overview_df
search_stages_df = diagnostics.search_stages_df
serp_results_df = diagnostics.serp_results_df
results_df = diagnostics.results_df
agentic_df = diagnostics.agentic_df
feature_evidence_df = diagnostics.feature_evidence_df
feature_matrix_df = diagnostics.feature_matrix_df
funnel_df = diagnostics.funnel_df
domain_summary_df = diagnostics.domain_summary_df
stage_quality_df = diagnostics.stage_quality_df
rejection_reasons_df = diagnostics.rejection_reasons_df
selection_rca_df = diagnostics.selection_rca_df
search_actions_df = adaptive_diagnostics.search_actions_df
search_engine_summary_df = adaptive_diagnostics.search_engine_summary_df
search_handles_df = adaptive_diagnostics.search_handles_df
search_decision_rca_df = adaptive_diagnostics.search_decision_rca_df
source_hierarchy_df = search_actions_df[[column for column in ('serp_credit', 'target_source_tier', 'market_stage', 'engine', 'purpose', 'planner_source', 'results_returned', 'new_candidate_urls', 'candidates_qualified', 'candidates_scraped', 'working_url_found', 'reason', 'query') if column in search_actions_df]].copy()
source_tier_summary_df = (results_df.groupby(['source_tier', 'source_tier_name', 'source_role'], dropna=False).agg(candidate_urls=('url', 'count'), scrape_attempted=('scrape_attempted', 'sum'), scrape_accepted=('scrape_success', 'sum'), identity_accepted=('identity_accepted', 'sum'), selected_urls=('strict_selected', 'sum'), mean_confidence=('confidence', 'mean')).reset_index().sort_values(['source_tier', 'mean_confidence'], ascending=[True, False]) if not results_df.empty and 'source_tier' in results_df else pd.DataFrame())
url_delivery_df = pd.DataFrame([{**(result.get('url_delivery') or {}), 'job_status': result.get('job_status'), 'strict_acceptance': (result.get('primary_url_acceptance') or {}).get('accepted'), 'primary_url': result.get('primary_url'), 'selection_scope': (result.get('product_match') or {}).get('selection_scope')}])
display_rich_summary(diagnostics)
display_adaptive_search_summary(adaptive_diagnostics)


## 4. Mandatory URL, market route, candidate RCA, and graphical diagnostics


In [ ]:
display_compact(url_delivery_df, title='Mandatory product URL delivery', max_rows=5)
display_compact(source_hierarchy_df, title='Source hierarchy by SerpAPI credit', max_rows=10)
display_compact(search_actions_df, title='Adaptive SerpAPI decisions', max_rows=10)
display_compact(search_engine_summary_df, title='Search-engine yield and conversion', max_rows=20)
display_compact(search_decision_rca_df, title='Search decision RCA', max_rows=40)
display_compact(search_handles_df, title='Product tokens, identifiers and image handles', max_rows=30)
display_compact(source_tier_summary_df, title='Candidate conversion by source tier', max_rows=20)
display_compact(results_df, title='One canonical product URL candidate per row', max_rows=100)
if not results_df.empty:
    assert results_df['url'].is_unique
display_compact(selection_rca_df, title='Final URL selection RCA', max_rows=30)
display_compact(funnel_df, title='Candidate conversion funnel', max_rows=20)
display_compact(rejection_reasons_df, title='Rejection and review reasons', max_rows=40)
display_compact(agentic_df, title='Agentic-browser investigations', max_rows=30)
display_compact(feature_evidence_df, title='URL-feature evidence', max_rows=100)
plot_engine_credit_allocation(adaptive_diagnostics)
plot_engine_candidate_yield(adaptive_diagnostics)
plot_credit_progression(adaptive_diagnostics)
plot_funnel(diagnostics)
plot_stage_yield(diagnostics)
plot_candidate_outcomes(diagnostics)
plot_confidence_distribution(diagnostics)
plot_confidence_vs_coverage(diagnostics)
plot_domain_quality(diagnostics)
plot_rejection_reasons(diagnostics)
plot_feature_heatmap(diagnostics)
plot_all_diagnostics(diagnostics)
print('FINAL URL TO OPEN IN BROWSER:')
print(result.get('primary_url'))


## 5. Export the complete review workbook


In [ ]:
tables = diagnostics.tables()
tables.update({'product_identification': product_identification_df, 'product_hypotheses': hypotheses_df, 'product_uncertainties': uncertainties_df, 'belief_updates': belief_updates_df, 'evidence_ledger': evidence_ledger_df, 'url_delivery': url_delivery_df, 'source_hierarchy': source_hierarchy_df, 'source_tier_summary': source_tier_summary_df})
workbook_path = artifact_path / 'single_product_diagnostics.xlsx'
with pd.ExcelWriter(workbook_path, engine='openpyxl') as writer:
    for sheet_name, frame in tables.items():
        if isinstance(frame, pd.DataFrame):
            frame.to_excel(writer, sheet_name=str(sheet_name)[:31], index=False)
export_adaptive_search_tables(adaptive_diagnostics, workbook_path)
print(f'Workbook: {workbook_path}')
print(f'Adaptive trace: {artifact_path / "adaptive_search_trace.json"}')
print(f'Mandatory delivery: {artifact_path / "mandatory_url_delivery.json"}')


## Reviewer checklist

Open the final URL and verify exact product identity, product form, model, variant, size, pack interpretation, information richness, and the reported market scope. A `REVIEW_REQUIRED` URL is a real review candidate, not an automated exact-match claim.
